# Experiment 1: Core Order-Sensitivity (D0 + D3 + D4)

D3 (swap halves) and D4 (full reverse) are the most decisive tests.
D0 (no-op) runs first on 3 docs as a sanity check.

Runs both Llama and Mistral sequentially. Caches per (corpus, probe, condition).

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import json, math, os, gc, random, time
from pathlib import Path
from scipy import stats
from tqdm.auto import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from google.colab import drive
drive.mount('/content/drive')

# ========================================================
# MODE SETTINGS — change these before running
# ========================================================
QUICK_MODE = True    # True = 10 shuffles, False = 50 (protocol)
PILOT_MODE = True    # True = 2 corpora + Llama only, False = all 8 + both probes
# ========================================================

tag = 'pilot' if PILOT_MODE else ('quick' if QUICK_MODE else 'full')
BASE = Path(f'/content/drive/MyDrive/LRTIA/Results/Exp1_disruption_{tag}')
BASE.mkdir(parents=True, exist_ok=True)

DATA = Path('/content/drive/MyDrive/LRTIA/Data')

ALL_CORPORA = {
    'wiki_zh': DATA / 'wiki_multilingual/zh_articles.jsonl',
    'wiki_ja': DATA / 'wiki_multilingual/ja_articles.jsonl',
    'wiki_ko': DATA / 'wiki_multilingual/ko_articles.jsonl',
    'wiki_tr': DATA / 'wiki_multilingual/tr_articles.jsonl',
    'wiki_ar': DATA / 'wiki_multilingual/ar_articles.jsonl',
    'wiki_fi': DATA / 'wiki_multilingual/fi_articles.jsonl',
    'buckeye': DATA / 'buckeye_processed/speaker_concatenated.jsonl',
    'french':  DATA / 'french_oral_processed/per_story.jsonl',
}

if PILOT_MODE:
    CORPORA = {k: v for k, v in ALL_CORPORA.items() if k in ('wiki_zh', 'buckeye')}
    PROBE_LIST = ['llama']
else:
    CORPORA = ALL_CORPORA
    PROBE_LIST = ['llama', 'mistral']

PROBES = {
    'llama': 'unsloth/Meta-Llama-3.1-8B',
    'mistral': 'mistralai/Mistral-7B-v0.1',
}

C = 100
TARGET_LEN = 30
TARGET_FRACS = [0.25, 0.50, 0.75]
MIN_BEFORE = C + 10
M = 50
N_SHUFFLES = 10 if QUICK_MODE else 50
SEED = 42

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'MODE: {"PILOT" if PILOT_MODE else "QUICK" if QUICK_MODE else "FULL"} ({N_SHUFFLES} shuffles)')
print(f'Corpora: {list(CORPORA.keys())}')
print(f'Probes: {PROBE_LIST}')
print(f'Results: {BASE}')

passes_per_target = C + 1 + C * N_SHUFFLES
targets_per_corpus = 60 * 3
ms_per_pass = 5
est_per_cond = passes_per_target * targets_per_corpus * ms_per_pass / 60000
n_combos = 3 * len(CORPORA) * len(PROBE_LIST)
print(f'Est. per corpus per condition: ~{est_per_cond:.0f} min')
print(f'Est. total ({n_combos} combos): ~{est_per_cond * n_combos:.0f} min = ~{est_per_cond * n_combos / 60:.1f} hrs')
print('Setup done')

In [ ]:
# === Disruption functions ===

def d0_no_op(ctx, M=50):
    near = ctx[-M:]
    far = ctx[:-M]
    return far + near

def d3_swap_halves(ctx, M=50):
    near = ctx[-M:]
    far = ctx[:-M]
    return near + far

def d4_full_reverse(ctx):
    return list(reversed(ctx))

# === Perplexity (matches existing pipeline exactly) ===

@torch.no_grad()
def compute_ppl(token_ids, target_start, target_end):
    if target_start >= target_end - 1:
        return float('inf')
    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    total_loss = 0.0
    count = 0
    for i in range(target_start, target_end - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        total_loss += -log_probs[token_ids[i + 1]].item()
        count += 1
    del outputs, logits
    torch.cuda.empty_cache()
    return math.exp(total_loss / count) if count > 0 else float('inf')

def compute_corrected_marginals(target_ids, disrupted_ctx, n_shuffles=N_SHUFFLES, seed=SEED):
    """Condition-specific corrected marginals.
    Returns dict with distances, ppl_ordered, ppl_shuffled, delta."""
    ctx_len = len(disrupted_ctx)
    rng = random.Random(seed)
    ppl_ord = []
    ppl_shuf = []
    
    for c in range(ctx_len + 1):
        if c == 0:
            chunk = list(target_ids)
            ppl = compute_ppl(chunk, 0, len(chunk))
            ppl_ord.append(ppl)
            ppl_shuf.append(ppl)
        else:
            prefix = disrupted_ctx[-c:]
            chunk = prefix + list(target_ids)
            ppl = compute_ppl(chunk, len(prefix), len(chunk))
            ppl_ord.append(ppl)
            # Shuffled baseline: shuffle same c-token multiset
            s_ppls = []
            for _ in range(n_shuffles):
                s_prefix = list(prefix)
                rng.shuffle(s_prefix)
                s_chunk = s_prefix + list(target_ids)
                s_ppl = compute_ppl(s_chunk, len(s_prefix), len(s_chunk))
                if not math.isinf(s_ppl):
                    s_ppls.append(s_ppl)
            ppl_shuf.append(np.mean(s_ppls) if s_ppls else ppl)
    
    distances = list(range(1, ctx_len + 1))
    m_ord = [ppl_ord[d-1] - ppl_ord[d] for d in distances]
    m_shuf = [ppl_shuf[d-1] - ppl_shuf[d] for d in distances]
    delta = [mo - ms for mo, ms in zip(m_ord, m_shuf)]
    
    return {
        'distances': distances,
        'ppl_ordered': ppl_ord,
        'ppl_shuffled': ppl_shuf,
        'm_ordered': m_ord,
        'm_shuffled': m_shuf,
        'delta': delta,
    }

def process_one_target(full_ids, target_start, target_end, condition_fn, condition_kwargs=None):
    """Process one target region under one disruption condition."""
    target_ids = full_ids[target_start:target_end]
    context = list(full_ids[max(0, target_start - C):target_start])
    if len(context) < C:
        return None
    
    if condition_kwargs:
        disrupted = condition_fn(context, **condition_kwargs)
    else:
        disrupted = condition_fn(context)
    
    return compute_corrected_marginals(target_ids, disrupted)

print('Functions defined')

In [ ]:
# === D0 Smoke Test: Llama only ===
# Quick mode: 2 docs, 1 target each. Full mode: 3 docs.
# Must produce curves indistinguishable from intact before we trust anything.

N_SMOKE_DOCS = 2 if QUICK_MODE else 3

print('='*60)
print(f'D0 SMOKE TEST ({N_SMOKE_DOCS} docs, Llama, Chinese Wiki, {N_SHUFFLES} shuffles)')
print('='*60)

tokenizer = AutoTokenizer.from_pretrained(PROBES['llama'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    PROBES['llama'],
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16),
    device_map='auto'
)
model.eval()
print('Model loaded')

# Load docs
corpus = []
with open(CORPORA['wiki_zh']) as f:
    for i, line in enumerate(f):
        if i >= N_SMOKE_DOCS: break
        corpus.append(json.loads(line))

d0_deltas = []
intact_deltas = []

t0 = time.time()
for doc in corpus:
    full_ids = tokenizer.encode(doc['text'], add_special_tokens=False)
    n = len(full_ids)
    target_start = int(n * 0.5)
    target_end = min(target_start + TARGET_LEN, n)
    if target_start < MIN_BEFORE:
        continue
    
    print(f'  {doc["doc_id"][:30]}: {n} tokens, target at {target_start}')
    
    intact_result = process_one_target(
        full_ids, target_start, target_end,
        lambda ctx: ctx
    )
    
    d0_result = process_one_target(
        full_ids, target_start, target_end,
        d0_no_op, {'M': M}
    )
    
    if intact_result and d0_result:
        intact_deltas.append(intact_result['delta'])
        d0_deltas.append(d0_result['delta'])
        max_diff = max(abs(a - b) for a, b in zip(intact_result['delta'], d0_result['delta']))
        mean_diff = np.mean([abs(a - b) for a, b in zip(intact_result['delta'], d0_result['delta'])])
        print(f'    D0 vs intact: max_diff={max_diff:.6f}, mean_diff={mean_diff:.6f}')

elapsed = time.time() - t0
print(f'\n  Smoke test took {elapsed:.0f}s ({elapsed/60:.1f} min)')
print(f'  Time per target: {elapsed / len(corpus):.0f}s')

if intact_deltas and d0_deltas:
    all_intact = np.mean(intact_deltas, axis=0)
    all_d0 = np.mean(d0_deltas, axis=0)
    overall_mean = np.mean(np.abs(all_intact - all_d0))
    intact_mag = np.mean(np.abs(all_intact))
    pct_diff = overall_mean / intact_mag * 100 if intact_mag > 0 else 0
    print(f'  As % of intact magnitude: {pct_diff:.2f}%')
    if pct_diff < 1:
        print('  D0 PASSES — indistinguishable from intact')
    elif pct_diff < 5:
        print('  D0 within 5% tolerance but not exact — check')
    else:
        print('  D0 FAILS — pipeline has a bug, do not proceed')
    
    # Also report timing extrapolation
    secs_per_target = elapsed / len(corpus)
    total_targets = sum(1 for _ in open(CORPORA['wiki_zh'])) * 3  # rough
    est_total = secs_per_target * total_targets * 3 * 8 * 2  # conditions * corpora * probes
    print(f'\n  TIMING EXTRAPOLATION:')
    print(f'    Per target (2 conditions, {N_SHUFFLES} shuffles): {secs_per_target:.0f}s')
    print(f'    Full run estimate: ~{est_total/3600:.0f} hours')

In [ ]:
# === Main loop: D3 + D4 on selected corpora and probes ===

CONDITIONS = {
    'intact': {'fn': lambda ctx: ctx, 'kwargs': {}},
    'D3_M50': {'fn': d3_swap_halves, 'kwargs': {'M': 50}},
    'D4': {'fn': d4_full_reverse, 'kwargs': {}},
}

def run_conditions_on_corpus(corpus_name, corpus_path, probe_name):
    """Run intact + D3 + D4 on one corpus. Cache per condition."""
    docs = []
    with open(corpus_path) as f:
        for line in f:
            docs.append(json.loads(line))
    
    for cond_name, cond_spec in CONDITIONS.items():
        cache_path = BASE / f'{probe_name}_{corpus_name}_{cond_name}.json'
        if cache_path.exists():
            with open(cache_path) as f:
                cached = json.load(f)
            print(f'  {cond_name}: cached ({len(cached)} results)')
            continue
        
        print(f'  {cond_name}: processing {len(docs)} docs...')
        t0 = time.time()
        results = []
        
        for doc in tqdm(docs, desc=f'{corpus_name}/{cond_name}'):
            full_ids = tokenizer.encode(doc['text'], add_special_tokens=False)
            n = len(full_ids)
            
            for frac in TARGET_FRACS:
                target_start = int(n * frac)
                target_end = min(target_start + TARGET_LEN, n)
                if target_start < MIN_BEFORE or target_end - target_start < 5:
                    continue
                
                r = process_one_target(
                    full_ids, target_start, target_end,
                    cond_spec['fn'],
                    cond_spec['kwargs'] if cond_spec['kwargs'] else None
                )
                if r:
                    r['doc_id'] = doc.get('doc_id', '')
                    r['target_frac'] = frac
                    results.append(r)
        
        elapsed = time.time() - t0
        with open(cache_path, 'w') as f:
            json.dump(results, f)
        print(f'    {len(results)} results in {elapsed/60:.1f} min')
        
        if results:
            mean_delta = np.mean([np.mean(r['delta']) for r in results])
            print(f'    Mean corrected marginal: {mean_delta:.4f}')

# === Run all probes ===
for probe_name in PROBE_LIST:
    print(f'\n{"="*60}')
    print(f'{probe_name.upper()}: D3 + D4 on {list(CORPORA.keys())}')
    print(f'{"="*60}')
    
    # Load model (skip if already loaded from smoke test and same probe)
    if probe_name == 'llama' and 'model' in dir() and model is not None:
        print('Llama already loaded from smoke test')
    else:
        if 'model' in dir() and model is not None:
            del model, tokenizer
            gc.collect()
            torch.cuda.empty_cache()
        
        tokenizer = AutoTokenizer.from_pretrained(PROBES[probe_name])
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        model = AutoModelForCausalLM.from_pretrained(
            PROBES[probe_name],
            quantization_config=BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type='nf4',
                bnb_4bit_compute_dtype=torch.float16),
            device_map='auto'
        )
        model.eval()
        print(f'{probe_name} loaded')
    
    for corpus_name, corpus_path in CORPORA.items():
        print(f'\n--- {corpus_name} ---')
        if not corpus_path.exists():
            print(f'  NOT FOUND: {corpus_path}')
            continue
        run_conditions_on_corpus(corpus_name, corpus_path, probe_name)
    
    # Print summary for this probe
    print(f'\n--- {probe_name.upper()} SUMMARY ---')
    print(f'{"Corpus":<15} {"Cond":<10} {"Mean Δ":>10} {"Detail":>30}')
    print('-' * 68)
    
    for corpus_name in CORPORA:
        for cond in ['intact', 'D3_M50', 'D4']:
            cache_path = BASE / f'{probe_name}_{corpus_name}_{cond}.json'
            if not cache_path.exists():
                continue
            with open(cache_path) as f:
                results = json.load(f)
            if not results:
                continue
            
            all_deltas = np.array([r['delta'] for r in results])
            mean_curve = np.mean(all_deltas, axis=0)
            mean_delta = np.mean(mean_curve)
            
            detail = ''
            if cond == 'D3_M50':
                jump = mean_curve[M] - mean_curve[M-1] if len(mean_curve) > M else 0
                pre_sd = np.std(mean_curve[:M]) if M > 0 else 1
                std_jump = jump / pre_sd if pre_sd > 0 else 0
                detail = f'jump={jump:.4f} (z={std_jump:.1f})'
            elif cond == 'D4':
                intact_path = BASE / f'{probe_name}_{corpus_name}_intact.json'
                if intact_path.exists():
                    with open(intact_path) as f:
                        intact_r = json.load(f)
                    intact_curve = np.mean([r['delta'] for r in intact_r], axis=0)
                    rev_intact = intact_curve[::-1]
                    rho_rev, _ = stats.spearmanr(mean_curve, rev_intact)
                    rho_fwd, _ = stats.spearmanr(mean_curve, intact_curve)
                    detail = f'rev_r={rho_rev:.3f} fwd_r={rho_fwd:.3f}'
            
            print(f'{corpus_name:<15} {cond:<10} {mean_delta:>10.4f} {detail:>30}')

print('\nAll probes done!')

In [ ]:
# === Quick visualization: D3 and D4 curves vs intact ===
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d

n_corpora = len(CORPORA)
ncols = min(4, n_corpora)
nrows = (n_corpora + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows), squeeze=False)
axes_flat = axes.flatten()

for idx, corpus_name in enumerate(CORPORA):
    ax = axes_flat[idx]
    
    for cond, color in [('intact', 'blue'), ('D3_M50', 'red'), ('D4', 'green')]:
        for probe in PROBE_LIST:
            cache_path = BASE / f'{probe}_{corpus_name}_{cond}.json'
            if not cache_path.exists():
                continue
            with open(cache_path) as f:
                results = json.load(f)
            if not results:
                continue
            mean_curve = np.mean([r['delta'] for r in results], axis=0)
            smooth = uniform_filter1d(mean_curve, 5)
            ax.plot(range(1, len(smooth)+1), smooth, color=color,
                    linewidth=2, label=f'{cond}', alpha=0.9)
            break  # first available probe
    
    ax.axvline(M, color='gray', linestyle=':', alpha=0.5, label=f'M={M}')
    ax.axhline(0, color='gray', linestyle=':', alpha=0.3)
    ax.set_title(corpus_name, fontweight='bold')
    ax.set_xlabel('Distance (tokens)')
    ax.set_ylabel('Corrected Marginal')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.15)

for idx in range(n_corpora, len(axes_flat)):
    axes_flat[idx].set_visible(False)

plt.suptitle('D3 (swap halves) and D4 (full reverse) vs Intact',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE / 'fig_D3_D4_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved')